In [ ]:
import numpy as np
import torch
import torch.nn as nn
import os
from google.colab import drive
drive.mount('/content/drive')

# Load the data
X_train = np.load("/content/drive/MyDrive/AI_Crime_Prediction/datasets/X_train.npy")
X_test = np.load("/content/drive/MyDrive/AI_Crime_Prediction/datasets/X_test.npy")
y_train = np.load("/content/drive/MyDrive/AI_Crime_Prediction/datasets/y_train.npy")
y_test = np.load("/content/drive/MyDrive/AI_Crime_Prediction/datasets/y_test.npy")

# Convert to PyTorch format
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)

# Handle the fact that some risk classes have more data than others
classes = np.unique(y_train.numpy())
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.numpy())
class_weights = torch.FloatTensor(class_weights)

In [ ]:
class CrimeLSTM(nn.Module):
    def __init__(self):
        super(CrimeLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, 3) # Output: 3 (Low, Medium, High)

    def forward(self, x):
        x = x.unsqueeze(-1)
        output, (hidden, cell) = self.lstm(x)
        hidden = hidden[-1]
        hidden = self.dropout(hidden)
        return self.fc(hidden)

model = CrimeLSTM()
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 20
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f}")

Epoch [1/20] - Loss: 0.8903
Epoch [2/20] - Loss: 0.8753
Epoch [3/20] - Loss: 0.8745
Epoch [4/20] - Loss: 0.8730
Epoch [5/20] - Loss: 0.8729
Epoch [6/20] - Loss: 0.8728
Epoch [7/20] - Loss: 0.8725
Epoch [8/20] - Loss: 0.8733
Epoch [9/20] - Loss: 0.8722
Epoch [10/20] - Loss: 0.8719
Epoch [11/20] - Loss: 0.8721
Epoch [12/20] - Loss: 0.8726
Epoch [13/20] - Loss: 0.8722
Epoch [14/20] - Loss: 0.8709
Epoch [15/20] - Loss: 0.8722
Epoch [16/20] - Loss: 0.8714
Epoch [17/20] - Loss: 0.8715
Epoch [18/20] - Loss: 0.8714
Epoch [19/20] - Loss: 0.8715
Epoch [20/20] - Loss: 0.8720


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/AI_Crime_Prediction/models"
os.makedirs(MODEL_PATH, exist_ok=True)
torch.save(model.state_dict(), f"{MODEL_PATH}/lstm_model.pth")
print("LSTM Model Saved Successfully to Google Drive!")

LSTM Model Saved Successfully to Google Drive!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        predictions.extend(predicted.numpy())
        actuals.extend(y_batch.numpy())

print("="*50)
print("FINAL MODEL EVALUATION")
print("="*50)
print(f"Accuracy: {accuracy_score(actuals, predictions):.4f}")
print("\nClassification Report:\n", classification_report(actuals, predictions))
print("\nConfusion Matrix:\n", confusion_matrix(actuals, predictions))

FINAL MODEL EVALUATION
Accuracy: 0.6743

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81      8569
           1       0.37      0.29      0.32      2963
           2       0.45      0.67      0.54      1386

    accuracy                           0.67     12918
   macro avg       0.54      0.59      0.56     12918
weighted avg       0.67      0.67      0.67     12918


Confusion Matrix:
 [[6930 1148  491]
 [1474  850  639]
 [ 169  287  930]]
